In [2]:

# Check whether a term is a variable
def is_variable(term):
    return term.isalpha() and term.islower()


# Parse an expression
def parse_atom(expression):
    predicate, arguments = expression.strip().split("(", 1)
    arguments = arguments.rstrip(")")
    args = tuple(x.strip() for x in arguments.split(","))
    return predicate.strip(), args


# Apply substitutions
def substitute(term, subs):
    while term in subs:
        term = subs[term]
    return term


# Unify two terms
def unify_terms(t1, t2, subs):
    t1 = substitute(t1, subs)
    t2 = substitute(t2, subs)

    if t1 == t2:
        return True

    if is_variable(t1):
        subs[t1] = t2
        return True

    if is_variable(t2):
        subs[t2] = t1
        return True

    return False


# Unify two predicates
def unify_atoms(a1, a2, subs):
    if a1[0] != a2[0] or len(a1[1]) != len(a2[1]):
        return None

    result = subs.copy()

    for t1, t2 in zip(a1[1], a2[1]):
        if not unify_terms(t1, t2, result):
            return None

    return result


# Apply substitutions to an expression
def apply_substitution(atom, subs):
    return (
        atom[0],
        tuple(substitute(x, subs) for x in atom[1])
    )


# Match rule conditions with facts
def match_body(body, kb, subs=None):
    if subs is None:
        subs = {}

    if not body:
        return [subs]

    results = []

    first = apply_substitution(body[0], subs)

    for fact in kb:
        new_subs = unify_atoms(first, fact, subs)

        if new_subs is not None:
            results.extend(
                match_body(body[1:], kb, new_subs)
            )

    return results


# Forward Chaining Algorithm
def forward_chaining(kb, rules, query):

    query = parse_atom(query)
    kb = [parse_atom(fact) for fact in kb]

    parsed_rules = []

    for rule in rules:
        left, right = rule.split("->")

        body = [
            parse_atom(x.strip())
            for x in left.split("&")
        ]

        head = parse_atom(right.strip())

        parsed_rules.append((body, head))

    print("\nInitial Knowledge Base:")

    for fact in kb:
        print(fact)

    print("\nApplying Forward Chaining...")

    # Repeat until no new facts are generated
    while True:

        new = []

        for body, head in parsed_rules:

            substitutions = match_body(body, kb)

            for subs in substitutions:

                inferred = apply_substitution(head, subs)

                if inferred not in kb and inferred not in new:
                    new.append(inferred)

        # No new facts generated
        if not new:
            print("\nQuery cannot be proved.")
            return False

        print("\nNew facts inferred:")

        for fact in new:
            print(fact)

        # Add new facts to KB
        kb.extend(new)

        # Check whether query is proved
        for fact in new:

            result = unify_atoms(query, fact, {})

            if result is not None:
                print("\nQuery proved!")
                print("Substitution:", result)
                print("Inferred fact:", fact)
                return result


# Main Program
print("FORWARD CHAINING IN FIRST ORDER LOGIC")

kb = [
    "Parent(John, Mary)",
    "Parent(Mary, Alice)"
]

rules = [
    "Parent(x, y) & Parent(y, z) -> Grandparent(x, z)"
]

print("\nKnowledge Base Rules:")
for rule in rules:
    print(rule)

query = input("\nEnter the query: ")

forward_chaining(kb, rules, query)


FORWARD CHAINING IN FIRST ORDER LOGIC

Knowledge Base Rules:
Parent(x, y) & Parent(y, z) -> Grandparent(x, z)

Enter the query: Grandparent(John, z)

Initial Knowledge Base:
('Parent', ('John', 'Mary'))
('Parent', ('Mary', 'Alice'))

Applying Forward Chaining...

New facts inferred:
('Grandparent', ('John', 'Alice'))

Query proved!
Substitution: {'z': 'Alice'}
Inferred fact: ('Grandparent', ('John', 'Alice'))


{'z': 'Alice'}